In [6]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

spark = SparkSession.builder \
    .appName("tugas11-penjualan") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.shuffle.partitions", "4")

print("SparkSession siap. Versi Spark:", spark.version)

SparkSession siap. Versi Spark: 3.5.9


In [8]:
import numpy as np
import pandas as pd

np.random.seed(111)
n = 2500
menu_list = ["Kopi Susu", "Americano", "Cappuccino", "Roti Bakar", "Kentang Goreng", "Es Teh"]
cabang_list = ["Magelang", "Yogyakarta", "Semarang"]
waktu_list = ["Pagi", "Siang", "Sore", "Malam"]

data_tugas11 = pd.DataFrame({
    "nama_menu": np.random.choice(menu_list, size=n),
    "cabang": np.random.choice(cabang_list, size=n),
    "waktu_pesan": np.random.choice(waktu_list, size=n),
    "qty": np.random.randint(1, 6, size=n),
    "harga_satuan": np.random.choice([15000, 18000, 20000, 22000, 25000, 30000], size=n),
})
data_tugas11["total_penjualan"] = data_tugas11["qty"] * data_tugas11["harga_satuan"] + np.random.normal(0, 8000, size=n)
data_tugas11.to_csv("tugas11_penjualan.csv", index=False)

print(f"Dataset berhasil dibuat: {len(data_tugas11)} baris")
print(data_tugas11.head())

df = spark.createDataFrame(data_tugas11)
df.show(5)
print("Total baris:", df.count())

Dataset berhasil dibuat: 2500 baris
        nama_menu      cabang waktu_pesan  qty  harga_satuan  total_penjualan
0  Kentang Goreng    Semarang       Malam    1         30000     27750.163532
1  Kentang Goreng  Yogyakarta        Sore    3         20000     64216.160744
2  Kentang Goreng    Semarang       Siang    5         22000    108722.792592
3  Kentang Goreng  Yogyakarta        Pagi    2         20000     54483.205872
4      Roti Bakar    Magelang       Malam    2         30000     58157.944120


+--------------+----------+-----------+---+------------+------------------+
|     nama_menu|    cabang|waktu_pesan|qty|harga_satuan|   total_penjualan|
+--------------+----------+-----------+---+------------+------------------+
|Kentang Goreng|  Semarang|      Malam|  1|       30000|27750.163531558544|
|Kentang Goreng|Yogyakarta|       Sore|  3|       20000| 64216.16074371864|
|Kentang Goreng|  Semarang|      Siang|  5|       22000|108722.79259154423|
|Kentang Goreng|Yogyakarta|       Pagi|  2|       20000|54483.205871792496|
|    Roti Bakar|  Magelang|      Malam|  2|       30000| 58157.94412015849|
+--------------+----------+-----------+---+------------+------------------+
only showing top 5 rows



Total baris: 2500


In [9]:
from pyspark.ml.feature import StringIndexer

indexer_nama_menu = StringIndexer(inputCol="nama_menu", outputCol="nama_menu_idx")
indexer_cabang = StringIndexer(inputCol="cabang", outputCol="cabang_idx")
indexer_waktu_pesan = StringIndexer(inputCol="waktu_pesan", outputCol="waktu_pesan_idx")

df_indexed = indexer_nama_menu.fit(df).transform(df)
df_indexed = indexer_cabang.fit(df_indexed).transform(df_indexed)
df_indexed = indexer_waktu_pesan.fit(df_indexed).transform(df_indexed)

df_indexed.select("nama_menu", "nama_menu_idx", "cabang", "cabang_idx", "waktu_pesan", "waktu_pesan_idx").show(8)

+--------------+-------------+----------+----------+-----------+---------------+
|     nama_menu|nama_menu_idx|    cabang|cabang_idx|waktu_pesan|waktu_pesan_idx|
+--------------+-------------+----------+----------+-----------+---------------+
|Kentang Goreng|          5.0|  Semarang|       2.0|      Malam|            1.0|
|Kentang Goreng|          5.0|Yogyakarta|       1.0|       Sore|            3.0|
|Kentang Goreng|          5.0|  Semarang|       2.0|      Siang|            2.0|
|Kentang Goreng|          5.0|Yogyakarta|       1.0|       Pagi|            0.0|
|    Roti Bakar|          4.0|  Magelang|       0.0|      Malam|            1.0|
|     Americano|          0.0|  Magelang|       0.0|       Pagi|            0.0|
|    Cappuccino|          2.0|  Magelang|       0.0|      Malam|            1.0|
|    Cappuccino|          2.0|  Semarang|       2.0|       Pagi|            0.0|
+--------------+-------------+----------+----------+-----------+---------------+
only showing top 8 rows



In [10]:
from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(
    inputCols=["qty", "harga_satuan", "nama_menu_idx", "cabang_idx", "waktu_pesan_idx"],
    outputCol="features"
)

df_final = assembler.transform(df_indexed)
df_final.select("features").show(5, truncate=False)

+-------------------------+
|features                 |
+-------------------------+
|[1.0,30000.0,5.0,2.0,1.0]|
|[3.0,20000.0,5.0,1.0,3.0]|
|[5.0,22000.0,5.0,2.0,2.0]|
|[2.0,20000.0,5.0,1.0,0.0]|
|[2.0,30000.0,4.0,0.0,1.0]|
+-------------------------+
only showing top 5 rows



In [11]:
train, test = df_final.randomSplit([0.75, 0.25], seed=101)

print(f"Jumlah data training : {train.count()}")
print(f"Jumlah data testing   : {test.count()}")

Jumlah data training : 1813
Jumlah data testing   : 687


In [12]:
from pyspark.ml.regression import LinearRegression

lr = LinearRegression(featuresCol="features", labelCol="total_penjualan")

model = lr.fit(train)

print("Model berhasil dilatih!")
print("Koefisien tiap fitur:", model.coefficients)
print("Intercept:", model.intercept)

Model berhasil dilatih!
Koefisien tiap fitur: [22118.89494914777,2.993923781437545,44.57221458380305,409.9146046922139,479.17189710040566]
Intercept: -67220.67040011783


In [14]:
predictions = model.transform(test)
predictions.select("qty", "harga_satuan", "nama_menu_idx", "cabang_idx", "waktu_pesan_idx","total_penjualan", "prediction").show(10)

+---+------------+-------------+----------+---------------+------------------+------------------+
|qty|harga_satuan|nama_menu_idx|cabang_idx|waktu_pesan_idx|   total_penjualan|        prediction|
+---+------------+-------------+----------+---------------+------------------+------------------+
|  4|       22000|          0.0|       0.0|            1.0|102174.09100317833| 87600.40448519964|
|  5|       22000|          0.0|       0.0|            1.0|115810.29600214415|109719.29943434741|
|  4|       20000|          0.0|       0.0|            0.0| 86874.15836101548| 81133.38502522414|
|  4|       25000|          0.0|       0.0|            0.0| 102614.5429236137| 96103.00393241188|
|  1|       18000|          0.0|       0.0|            2.0| 19477.97002961815| 9747.196409106575|
|  2|       30000|          0.0|       0.0|            2.0|  66900.3836581082| 67793.17673550487|
|  3|       15000|          0.0|       0.0|            2.0| 37337.34201517915| 45003.21496308947|
|  4|       18000|  

In [15]:
from pyspark.ml.evaluation import RegressionEvaluator

evaluator_rmse = RegressionEvaluator(labelCol="total_penjualan", predictionCol="prediction", metricName="rmse")
rmse = evaluator_rmse.evaluate(predictions)

evaluator_r2 = RegressionEvaluator(labelCol="total_penjualan", predictionCol="prediction", metricName="r2")
r2 = evaluator_r2.evaluate(predictions)

print(f"RMSE : Rp {rmse:,.0f}")
print(f"R²   : {r2:.4f}  ({r2*100:.1f}% variasi data berhasil dijelaskan model)")

RMSE : Rp 10,388
R²   : 0.9163  (91.6% variasi data berhasil dijelaskan model)


Model ini sudah baik untuk digunakan memprediksi omzet total penjualan. Nilai R² sebesar 0.9163 menunjukkan kalau 91,6% variasi data total penjualan bisa dijelaskan secara akurat oleh fitur yang digunakan, sedangkan rata-rata selisih prediksi dengan nilai aslinya hanya sebesar Rp 10.388

In [18]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.evaluation import RegressionEvaluator

assembler_eksp = VectorAssembler(
    inputCols=["qty", "harga_satuan"],
    outputCol="features_eksp"
)

df_eksp = assembler_eksp.transform(df)
train_eksp, test_eksp = df_eksp.randomSplit([0.75, 0.25], seed=101)

lr_eksp = LinearRegression(featuresCol="features_eksp", labelCol="total_penjualan")
model_eksp = lr_eksp.fit(train_eksp)

predictions_eksp = model_eksp.transform(test_eksp)
r2_eksp = evaluator_r2.evaluate(predictions_eksp)

print(f"Model Utama: {r2:.4f}")
print(f"Model eksperimen: {r2_eksp:.4f}")

Model Utama: 0.9163
Model eksperimen: 0.9169


Analisis :
Nilai R²  kedua model hampir sama, tetapi model ekperimen lebih tinggi sedikit. Ini menunjukkan kalau variasi nilai total penjualan hampir semuanya di bantu oleh kolom qty dan harga satuan.

Kesimpulan:
Menambahkan informasi nama_menu, cabang, dan waktu_pesan tidak meningkatkan akurasi prediksi secara berarti. Jal ini di buktikan dengan nilai R² model eksperimen yang hampir sama persis dengan model utama. Fitur yang paling dominan dan menentukan dalam memprediksi total penjualan hanyalah kombinasi perkalian antara kuantitas (qty) dan harga satuan

In [21]:
data_dummy = [
    ("Kopi Susu", "Magelang", "Pagi", 3, 15000),
    ("Americano", "Yogyakarta", "Siang", 2, 18000),
    ("Cappuccino", "Semarang", "Malam", 5, 22000)
]
kolom_dummy = ["nama_menu", "cabang", "waktu_pesan", "qty", "harga_satuan"]
df_baru = spark.createDataFrame(data_dummy, kolom_dummy)

df_baru_indexed = indexer_nama_menu.fit(df).transform(df_baru)
df_baru_indexed = indexer_cabang.fit(df).transform(df_baru_indexed)
df_baru_indexed = indexer_waktu_pesan.fit(df).transform(df_baru_indexed)

df_baru_features = assembler.transform(df_baru_indexed)

prediksi_final = model.transform(df_baru_features)

print("--- Bagian E: Hasil Prediksi 3 Skenario Baru ---")
prediksi_final.select("nama_menu", "cabang", "waktu_pesan", "qty", "harga_satuan", "prediction").show(truncate=False)

--- Bagian E: Hasil Prediksi 3 Skenario Baru ---
+----------+----------+-----------+---+------------+------------------+
|nama_menu |cabang    |waktu_pesan|qty|harga_satuan|prediction        |
+----------+----------+-----------+---+------------+------------------+
|Kopi Susu |Magelang  |Pagi       |3  |15000       |44089.44338347246 |
|Americano |Yogyakarta|Siang      |2  |18000       |32276.005962946554|
|Cappuccino|Semarang  |Malam      |5  |22000       |110628.27307289945|
+----------+----------+-----------+---+------------+------------------+



In [22]:
spark.stop()
print("SparkSession ditutup.")

SparkSession ditutup.
